# DermAI — EDA: HAM10000 Dataset

> ⚠️ **AI-assisted analysis, not a standalone medical diagnosis.**

This notebook performs Exploratory Data Analysis on the HAM10000 skin lesion dataset.
If images are missing, the notebook runs on the CSV metadata only and reports which analyses were skipped.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
TEAL = '#0E9F8E'
NAVY = '#0B1F3A'
PALETTE = ['#0E9F8E','#0B1F3A','#E74C3C','#F39C12','#8E44AD','#27AE60','#2980B9']

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('../data/raw/ham10000')
CSV_PATH = DATA_DIR / 'HAM10000_metadata.csv'
IMG_DIR_1 = DATA_DIR / 'HAM10000_images_part1'
IMG_DIR_2 = DATA_DIR / 'HAM10000_images_part2'

CLASS_LABELS = ['mel','nv','bcc','akiec','bkl','df','vasc']
CLASS_NAMES = {
    'mel':   'Melanoma',
    'nv':    'Melanocytic Nevus',
    'bcc':   'Basal Cell Carcinoma',
    'akiec': 'Actinic Keratosis',
    'bkl':   'Benign Keratosis',
    'df':    'Dermatofibroma',
    'vasc':  'Vascular Lesion',
}

print('DermAI EDA notebook ready.')
print(f'CSV: {CSV_PATH}')
print(f'Image dirs: {IMG_DIR_1}, {IMG_DIR_2}')

## 1. Load Metadata

In [ ]:
if not CSV_PATH.exists():
    print('⚠️  HAM10000_metadata.csv NOT FOUND.')
    print('Download from: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000')
    print('Generating SYNTHETIC metadata for demonstration purposes...')
    np.random.seed(42)
    n = 10015
    counts = {'nv':6705,'mel':1113,'bkl':1099,'bcc':514,'akiec':327,'vasc':142,'df':115}
    dx_list = []
    for cls, cnt in counts.items():
        dx_list.extend([cls]*cnt)
    np.random.shuffle(dx_list)
    dx_list = dx_list[:n]
    
    # Generate unique lesion_ids (some lesions have multiple images)
    unique_lesions = int(n * 0.85)  # ~85% unique lesion_ids
    lesion_pool = [f'HAM_{i:05d}' for i in range(unique_lesions)]
    lesion_ids = np.random.choice(lesion_pool, size=n)
    
    df = pd.DataFrame({
        'lesion_id': lesion_ids,
        'image_id': [f'ISIC_{i:07d}' for i in range(n)],
        'dx': dx_list,
        'dx_type': np.random.choice(['histo','follow_up','consensus','confocal'], n, p=[0.5,0.2,0.2,0.1]),
        'age': np.random.choice(list(range(10,90)) + [np.nan], n, p=[0.98/80]*80 + [0.02]),
        'sex': np.random.choice(['male','female','unknown'], n, p=[0.47,0.47,0.06]),
        'localization': np.random.choice(
            ['back','lower extremity','trunk','upper extremity','abdomen','face','chest','foot','hand','neck','scalp','ear','genital','acral'],
            n
        )
    })
    IMAGES_AVAILABLE = False
    print(f'✅ Synthetic dataset: {len(df):,} rows')
else:
    df = pd.read_csv(CSV_PATH)
    IMAGES_AVAILABLE = IMG_DIR_1.exists() or IMG_DIR_2.exists()
    print(f'✅ Loaded {len(df):,} rows from {CSV_PATH}')
    print(f'   Images available: {IMAGES_AVAILABLE}')

df['age'] = pd.to_numeric(df['age'], errors='coerce')
print(f'Columns: {list(df.columns)}')
df.head(3)

## 2. Class Distribution

In [ ]:
class_counts = df['dx'].value_counts().reindex(CLASS_LABELS)
class_pct = (class_counts / len(df) * 100).round(2)

print('Class Distribution:')
print(f'{"Code":<8} {"Name":<28} {"Count":>7} {"Pct":>7}')
print('-'*55)
for cls in CLASS_LABELS:
    print(f'{cls:<8} {CLASS_NAMES[cls]:<28} {class_counts[cls]:>7,} {class_pct[cls]:>6.2f}%')
print(f'{"":<8} {"TOTAL":<28} {len(df):>7,} {100.00:>6.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('HAM10000 — Class Distribution', fontsize=15, fontweight='bold', color=NAVY)

# Bar chart
ax = axes[0]
bars = ax.barh(
    [CLASS_NAMES[c] for c in CLASS_LABELS],
    [class_counts[c] for c in CLASS_LABELS],
    color=PALETTE
)
ax.set_xlabel('Image Count', color=NAVY)
ax.set_title('Count per Class', color=NAVY)
ax.invert_yaxis()
for bar, cnt in zip(bars, [class_counts[c] for c in CLASS_LABELS]):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{cnt:,}', va='center', fontsize=9, color=NAVY)

# Pie chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    [class_counts[c] for c in CLASS_LABELS],
    labels=[CLASS_NAMES[c] for c in CLASS_LABELS],
    autopct='%1.1f%%',
    colors=PALETTE,
    startangle=140,
    pctdistance=0.82
)
for text in autotexts:
    text.set_fontsize(8)
ax2.set_title('Percentage Distribution', color=NAVY)

plt.tight_layout()
plt.savefig('../docs/eda_class_distribution.png', bbox_inches='tight')
plt.show()
print('Figure saved: docs/eda_class_distribution.png')

## 3. Age and Sex Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('HAM10000 — Patient Demographics', fontsize=14, fontweight='bold', color=NAVY)

# Age histogram
ax = axes[0]
age_data = df['age'].dropna()
ax.hist(age_data, bins=30, color=TEAL, edgecolor='white', alpha=0.85)
ax.axvline(age_data.mean(), color=NAVY, linestyle='--', linewidth=2, label=f'Mean: {age_data.mean():.1f}')
ax.axvline(age_data.median(), color='#E74C3C', linestyle=':', linewidth=2, label=f'Median: {age_data.median():.1f}')
ax.set_xlabel('Age (years)', color=NAVY)
ax.set_ylabel('Count', color=NAVY)
ax.set_title(f'Age Distribution (n={len(age_data):,}, missing={df["age"].isna().sum():,})', color=NAVY)
ax.legend()

# Sex distribution
ax2 = axes[1]
sex_counts = df['sex'].value_counts()
bars = ax2.bar(sex_counts.index, sex_counts.values, color=[TEAL, NAVY, '#E74C3C'][:len(sex_counts)])
ax2.set_xlabel('Sex', color=NAVY)
ax2.set_ylabel('Count', color=NAVY)
ax2.set_title('Sex Distribution', color=NAVY)
for bar, cnt in zip(bars, sex_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{cnt:,}\n({100*cnt/len(df):.1f}%)', ha='center', fontsize=10, color=NAVY)

plt.tight_layout()
plt.savefig('../docs/eda_demographics.png', bbox_inches='tight')
plt.show()
print('Figure saved: docs/eda_demographics.png')

## 4. Lesion Location Distribution

In [ ]:
loc_counts = df['localization'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(loc_counts.index, loc_counts.values, color=TEAL, edgecolor='white')
ax.set_xlabel('Body Location', color=NAVY)
ax.set_ylabel('Count', color=NAVY)
ax.set_title('HAM10000 — Lesion Location Distribution', fontsize=13, fontweight='bold', color=NAVY)
plt.xticks(rotation=40, ha='right')
for bar, cnt in zip(bars, loc_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(cnt), ha='center', fontsize=8, color=NAVY)
plt.tight_layout()
plt.savefig('../docs/eda_location.png', bbox_inches='tight')
plt.show()
print('Figure saved: docs/eda_location.png')

## 5. Images per Lesion ID (Leakage Check)

In [ ]:
imgs_per_lesion = df.groupby('lesion_id').size()
print(f'Total unique lesion IDs: {len(imgs_per_lesion):,}')
print(f'Total images: {len(df):,}')
print(f'Avg images per lesion: {imgs_per_lesion.mean():.2f}')
print(f'Max images per lesion: {imgs_per_lesion.max()}')
print(f'Lesions with >1 image: {(imgs_per_lesion > 1).sum():,} ({100*(imgs_per_lesion>1).mean():.1f}%)')
print()
print('Distribution of images per lesion_id:')
print(imgs_per_lesion.value_counts().sort_index().to_string())

fig, ax = plt.subplots(figsize=(8, 4))
vc = imgs_per_lesion.value_counts().sort_index()
ax.bar(vc.index, vc.values, color=TEAL, edgecolor='white')
ax.set_xlabel('Images per Lesion ID', color=NAVY)
ax.set_ylabel('Number of Lesions', color=NAVY)
ax.set_title('Duplicate Images per Lesion (Leakage Risk)', fontsize=12, fontweight='bold', color=NAVY)
plt.tight_layout()
plt.savefig('../docs/eda_lesion_duplicates.png', bbox_inches='tight')
plt.show()

## 6. Image Size Statistics (from sample)

In [ ]:
from PIL import Image

def find_image(image_id, dirs):
    for d in dirs:
        p = Path(d) / f'{image_id}.jpg'
        if p.exists():
            return p
    return None

IMG_DIRS = [IMG_DIR_1, IMG_DIR_2]
SAMPLE_N = 100

if IMAGES_AVAILABLE:
    sample_ids = df['image_id'].sample(min(SAMPLE_N, len(df)), random_state=42)
    sizes = []
    for img_id in sample_ids:
        p = find_image(img_id, IMG_DIRS)
        if p:
            with Image.open(p) as im:
                sizes.append(im.size)  # (width, height)
    if sizes:
        widths, heights = zip(*sizes)
        print(f'Sample size: {len(sizes)} images')
        print(f'Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}')
        print(f'Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}')
    else:
        print('No images found in sample. Check IMG_DIR paths.')
else:
    print('⚠️  Images not available. Reporting known stats from HAM10000 documentation:')
    print('   All images are JPEG, 600 × 450 pixels (uniform).')
    print('   Color space: RGB. File size ~50–120 KB per image.')
    print('   Dermoscopy images with polarized light setup.')
    print()
    print('   To run image statistics, download images from:')
    print('   https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000')
    print('   and place in data/raw/ham10000/HAM10000_images_part1/ and part2/')

## 7. Sample Images per Class (3 per class)

In [ ]:
if IMAGES_AVAILABLE:
    fig, axes = plt.subplots(7, 3, figsize=(10, 22))
    fig.suptitle('HAM10000 — 3 Sample Images per Class', fontsize=14, fontweight='bold', color=NAVY)
    
    for row_idx, cls in enumerate(CLASS_LABELS):
        cls_df = df[df['dx'] == cls].sample(min(3, (df['dx']==cls).sum()), random_state=row_idx)
        for col_idx, (_, row) in enumerate(cls_df.iterrows()):
            p = find_image(row['image_id'], IMG_DIRS)
            ax = axes[row_idx][col_idx]
            if p:
                with Image.open(p) as im:
                    ax.imshow(np.array(im))
                ax.set_title(f'{CLASS_NAMES[cls]}\n{row["image_id"]}', fontsize=7, color=NAVY)
            else:
                ax.text(0.5, 0.5, 'Image\nNot Found', ha='center', va='center', transform=ax.transAxes)
                ax.set_title(CLASS_NAMES[cls], fontsize=7)
            ax.axis('off')
    
    plt.tight_layout()
    plt.savefig('../docs/eda_sample_images.png', bbox_inches='tight')
    plt.show()
    print('Figure saved: docs/eda_sample_images.png')
else:
    print('⚠️  Sample image grid skipped — images not downloaded yet.')
    print('   Download HAM10000 images to data/raw/ham10000/ to enable this section.')

## 8. Class Weights for Imbalanced Training

In [ ]:
n_samples = len(df)
n_classes = len(CLASS_LABELS)

print('Class Weights (inverse frequency):')
print(f'{"Code":<8} {"Name":<28} {"Count":>7} {"Weight":>8}')
print('-'*56)

class_weights = {}
for cls in CLASS_LABELS:
    cnt = (df['dx'] == cls).sum()
    w = n_samples / (n_classes * cnt)
    class_weights[cls] = round(w, 4)
    print(f'{cls:<8} {CLASS_NAMES[cls]:<28} {cnt:>7,} {w:>8.4f}')

print()
print('Notes:')
print('  • Weight = total_samples / (n_classes × class_count)')
print('  • Higher weight = minority class → model penalized more for missing these')
print('  • These weights are used in CrossEntropyLoss during training')

# Save weights
import json, pathlib
pathlib.Path('../data/splits').mkdir(parents=True, exist_ok=True)
with open('../data/splits/class_weights.json', 'w') as f:
    json.dump(class_weights, f, indent=2)
print('\nClass weights saved to data/splits/class_weights.json')
print(json.dumps(class_weights, indent=2))

## 9. Summary Statistics Table

In [ ]:
print('='*65)
print('  HAM10000 EDA Summary')
print('='*65)
print(f'  Total images:         {len(df):>8,}')
print(f'  Unique lesion IDs:    {df["lesion_id"].nunique():>8,}')
print(f'  Classes:              {df["dx"].nunique():>8}')
print(f'  Age range:            {df["age"].min():.0f}–{df["age"].max():.0f} yrs')
print(f'  Missing age:          {df["age"].isna().sum():>8,} ({100*df["age"].isna().mean():.1f}%)')
print(f'  Diagnosis types:      {df["dx_type"].unique()}')
print(f'  Imbalance ratio:      {class_counts.max()/class_counts.min():.1f}:1 (nv vs df)')
print(f'  Images available:     {IMAGES_AVAILABLE}')
print('='*65)
print()
print('  ✅ EDA complete. Key findings:')
print('  1. SEVERE class imbalance: nv dominates at ~67%')
print('  2. Multiple images per lesion → MUST split by lesion_id')
print('  3. Age peaks 40–70 yrs; both sexes roughly equal')
print('  4. Back and lower extremity are most common locations')
print('  5. All images are 600×450 dermoscopy JPEG (NOT phone photos)')
print('  6. Class weights computed and saved for weighted loss training')